# Silver: Câmbio (dim_cambio)

**Objetivo:** manter um histórico diário das taxas de câmbio (BRL, ARS, MXN → USD), consultando uma API pública, com estratégia de fallback para garantir resiliência caso a API falhe.

**Fonte:** API pública [exchangerate-api.com](https://www.exchangerate-api.com/) (testada e validada na etapa de conectividade do projeto).

**Destino:** tabela Delta `silver.dim_cambio`.

**Estratégia de resiliência (fallback):** se a chamada à API falhar (timeout, erro de status, rate limit), a última cotação disponível na própria tabela é reaproveitada para o dia corrente, marcada com `fonte_atualizada = false` — garantindo que o pipeline não seja interrompido por instabilidade de uma dependência externa.

**Granularidade:** histórico completo — uma linha por moeda, por dia, permitindo consultar a cotação vigente em qualquer data passada.

In [0]:
# Imports

import requests
from datetime import date, datetime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, BooleanType, TimestampType
from pyspark.sql.functions import col
from delta.tables import DeltaTable

print("Imports realizados com sucesso.")

In [0]:
# Chamada à API de câmbio

url_api_cambio = "https://api.exchangerate-api.com/v4/latest/USD"

api_funcionou = False
taxas_hoje = {}

try:
    response = requests.get(url_api_cambio, timeout=10)
    if response.status_code == 200:
        dados = response.json()
        taxas_hoje = {
            "BRL": dados["rates"]["BRL"],
            "ARS": dados["rates"]["ARS"],
            "MXN": dados["rates"]["MXN"],
        }
        api_funcionou = True
        print(f"API respondeu com sucesso. Taxas: {taxas_hoje}")
    else:
        print(f"API retornou status inesperado: {response.status_code}. Fallback será aplicado.")
except Exception as e:
    print(f"Erro ao chamar a API: {e}. Fallback será aplicado.")

In [0]:
# Montagem do DataFrame de câmbio (com fallback)

schema_cambio = StructType([
    StructField("moeda", StringType(), True),
    StructField("taxa_para_usd", DoubleType(), True),
    StructField("data_cotacao", DateType(), True),
    StructField("fonte_atualizada", BooleanType(), True),
    StructField("data_execucao", TimestampType(), True),
])

tabela_cambio_existe = spark.catalog.tableExists("poc_latam_food.silver.dim_cambio")

if api_funcionou:
    linhas_cambio = [
        (moeda, taxa, date.today(), True, datetime.now())
        for moeda, taxa in taxas_hoje.items()
    ]
    print("Usando taxas obtidas da API nesta execução.")

else:
    if not tabela_cambio_existe:
        raise Exception(
            "API falhou e não existe nenhuma cotação histórica em dim_cambio para aplicar fallback. "
            "Não é possível prosseguir na primeira execução sem a API disponível."
        )

    df_ultimas_taxas = (
        spark.table("poc_latam_food.silver.dim_cambio")
        .orderBy(col("data_cotacao").desc())
        .dropDuplicates(["moeda"])
        .select("moeda", "taxa_para_usd")
        .collect()
    )

    linhas_cambio = [
        (row["moeda"], row["taxa_para_usd"], date.today(), False, datetime.now())
        for row in df_ultimas_taxas
    ]
    print("API indisponível. Reaproveitando últimas cotações conhecidas (fallback).")

df_cambio_hoje = spark.createDataFrame(linhas_cambio, schema=schema_cambio)
df_cambio_hoje.display()

In [0]:
# Gravação com MERGE (evita duplicar cotação do mesmo dia)

if not tabela_cambio_existe:
    df_cambio_hoje.write.format("delta").saveAsTable("poc_latam_food.silver.dim_cambio")
    print("Tabela dim_cambio criada com a primeira carga.")

else:
    tabela_cambio = DeltaTable.forName(spark, "poc_latam_food.silver.dim_cambio")

    (
        tabela_cambio.alias("target")
        .merge(
            df_cambio_hoje.alias("source"),
            "target.moeda = source.moeda AND target.data_cotacao = source.data_cotacao"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("MERGE em dim_cambio concluído.")

In [0]:
# Validação visual - silver.dim_cambio

spark.table("poc_latam_food.silver.dim_cambio").display()

print(f"Total de registros: {spark.table('poc_latam_food.silver.dim_cambio').count()}")